In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from simulation.eg_parameters import Parameters
from simulation.eg_runner_usingMonitoredResource import RunnerMR
#from simulation.eg_model import Model
import os
from IPython.display import HTML
from itables import to_html_datatable
from simulation.eg_warmupAuditor import WarmupAuditor
from simulation.eg_model_usingMonitoredResource import ModelMR

def run_warmup_analysis(
    audit_interval, data_collection_period, number_of_runs
):
    """
    Runs the model without a warm-up, collecting performance measures at
    regular intervals, used to determine appropriate length of warm-up.

    Parameters
    ----------
    audit_interval : int
        Audit frequency (minutes).
    data_collection_period : int
        Duration of the data collection period (minutes).
    number_of_runs : int
        The number of runs (i.e., replications).

    Returns
    -------
    pd.DataFrame
        Combined results from all replications.
    """
    # Fixed set of parameters for all runs
    # Important: Sets warm-up period to zero
    param = Parameters(
        warm_up_period=0,
        data_collection_period=data_collection_period
    )
    # Store results from each replication
    all_dfs = []
    # Loop through replications
    for run in range(number_of_runs):
        # Run model with auditor
        model = ModelMR(param=param, run_number=run)
        auditor = WarmupAuditor(model=model, interval=audit_interval)
        auditor.run()
        # Convert the run's audit results to DataFrame
        run_df = pd.DataFrame(auditor.audit_results)
        run_df["run"] = run
        all_dfs.append(run_df)
    # Combine all runs into a single DataFrame
    return pd.concat(all_dfs, ignore_index=True)

In [2]:
audit_results = run_warmup_analysis(
    audit_interval=10,
    data_collection_period=14400,
    number_of_runs=10
)
# Preview audit results
HTML(to_html_datatable(audit_results.head(200)))

Loading ITables v2.6.2 from the internet... (need help?)


In [3]:
import plotly.express as px

def plot_time_series(audit_results, metric, warmup_time=None):
    """
    Plot cumulative mean trajectories and compute overall cumulative mean.

    Parameters
    ----------
    audit_results : pd.DataFrame
        Audit results, where columns include "time", "run" and the
        chosen metric like "wait_time" or "time_in_system".
    metric : str
        The performance measure to visualise.
    warmup_time : int
        The time point at which to display a vertical dashed line marking the
        suggested warm-up period.

    Returns
    -------
    plotly.graph_objects.Figure
        A Plotly Figure object containing cumulative mean trajectories for
        each run and the overall cumulative mean.
    """
    # Computer overall cumulative mean
    df = audit_results.groupby("time")[metric].mean().reset_index()
    df["overall_cumulative"] = df[metric].expanding().mean()

    # Plot cumulative mean for each run
    fig = px.line(
        data_frame=audit_results,
        x="time",
        y=metric,
        line_group="run"
    )
    fig.update_traces(line_color="lightblue")

    # Overlay overall cumulative mean
    overall_fig = px.line(df, x="time", y="overall_cumulative")
    fig.add_traces(list(overall_fig.select_traces()))

    # Add warm-up line
    if warmup_time:
        fig.add_vline(x=warmup_time, line_color="red", line_dash="dash",
                      annotation_text="Suggested warm-up",
                      annotation_font_color="red",
                      annotation_position="top right")

    # Axis labels and layout
    fig.update_layout(
        xaxis_title="Run time (minutes)",
        yaxis_title=f"cumulative_mean_{metric}",
        template="plotly_white"
    )
    return fig

In [4]:
plot_time_series(audit_results, "utilisation", 3000)

In [5]:
plot_time_series(audit_results, "wait_time", 3000)

In [6]:
plot_time_series(audit_results, "queue_length", 3000)

In [7]:
plot_time_series(audit_results, "patients_in_system", 3000)

In [8]:
plot_time_series(audit_results, "time_in_system", 3000)